In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
combined_data = pd.read_csv('./data/combined_ncaa.csv')

In [ ]:
combined_data.head()

,YEAR,TEAM ID,TEAM,CONF,SEED,ROUND,W,L,win_pct,KADJ EM,...,three_pct_net,three_rate_net,three_value,K TEMPO,FT%,FTR,PPPO,GAMES,3PA,3PM
0,2026,2,Akron,MAC,12,0,27,5,84.375000,12.7986,...,3.4,1.9,52.0905,71.2230,75.3,29.1,1.219,32,902,347
1,2026,3,Alabama,SEC,4,0,23,9,71.875000,25.7196,...,2.3,18.6,57.6738,74.6151,76.5,37.2,1.220,32,1123,402
2,2026,8,Arizona,B12,1,0,32,2,94.117647,37.6556,...,4.6,-10.4,28.9440,71.3814,73.4,42.9,1.203,34,552,199
3,2026,10,Arkansas,SEC,4,0,26,8,76.470588,26.0527,...,7.2,-4.8,38.9778,72.4168,74.7,36.4,1.225,34,719,280
4,2026,25,BYU,B12,6,0,23,11,67.647059,23.2459,...,-0.4,0.6,41.6706,70.4428,74.6,34.2,1.187,34,839,293


# NCAA Exploratory Data Analysis
## Hypothesis: Does 3-Point Shooting Affect Team Success?

**Focus:**
1. How has 3-point shooting changed over time?
2. Does 3-point shooting separate tournament teams from the rest?

## Section 1: Temporal Trends

How has 3-point shooting evolved over time — and has it become more or less predictive of win rate?

In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
from scipy import stats

# --- Graph 1: 3PT% vs Win Rate for Each Year ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, col, xlabel in zip(axes, ['3PT%', '3PTR'], ['3PT FG%', '3PT Attempt Rate (%)']):
    years = sorted(combined_data['YEAR'].unique())
    norm  = plt.Normalize(min(years), max(years))
    cmap  = plt.cm.viridis

    for year in years:
        sub = combined_data[combined_data['YEAR'] == year]
        color = cmap(norm(year))
        slope, intercept, r, *_ = stats.linregress(sub[col], sub['win_pct'])
        x_line = np.linspace(sub[col].min(), sub[col].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=1.4, alpha=0.8)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Year')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Win Rate (%)')
    ax.set_title(f'{xlabel} vs Win Rate — regression line per year')

plt.suptitle('How Does the 3PT → Win Rate Relationship Change Each Year?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Graph 2: 3PT Shooting vs Win Rate Over Time (Tournament vs Non-Tournament) ---
yearly_split = combined_data.groupby(['YEAR', 'made_tournament'])[['3PTR', '3PT%', 'win_pct']].mean().reset_index()
tourney     = yearly_split[yearly_split['made_tournament'] == 1]
non_tourney = yearly_split[yearly_split['made_tournament'] == 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: 3PTR over time
ax = axes[0]
ax.plot(tourney['YEAR'],     tourney['3PTR'],     marker='o', color='steelblue', linewidth=2, label='Tournament')
ax.plot(non_tourney['YEAR'], non_tourney['3PTR'], marker='s', color='coral',     linewidth=2, label='Non-Tournament')
ax.set_xlabel('Year')
ax.set_ylabel('Avg 3PT Attempt Rate (%)')
ax.set_title('3PTR Over Time')
ax.legend(fontsize=9)

# Panel 2: 3PT% over time
ax = axes[1]
ax.plot(tourney['YEAR'],     tourney['3PT%'],     marker='o', color='steelblue', linewidth=2, label='Tournament')
ax.plot(non_tourney['YEAR'], non_tourney['3PT%'], marker='s', color='coral',     linewidth=2, label='Non-Tournament')
ax.set_xlabel('Year')
ax.set_ylabel('Avg 3PT FG%')
ax.set_title('3PT% Over Time')
ax.legend(fontsize=9)

# Panel 3: avg 3PT% vs avg win_pct per year (scatter — each point = 1 year)
ax = axes[2]
ax.scatter(tourney['3PT%'],     tourney['win_pct'],     color='steelblue', s=50, label='Tournament',     zorder=3)
ax.scatter(non_tourney['3PT%'], non_tourney['win_pct'], color='coral',     s=50, label='Non-Tournament', zorder=3)
for _, row in tourney.iterrows():
    ax.annotate(str(int(row['YEAR'])), (row['3PT%'], row['win_pct']), fontsize=6, alpha=0.7)
for _, row in non_tourney.iterrows():
    ax.annotate(str(int(row['YEAR'])), (row['3PT%'], row['win_pct']), fontsize=6, alpha=0.7)
ax.set_xlabel('Avg 3PT FG%')
ax.set_ylabel('Avg Win Rate (%)')
ax.set_title('3PT% vs Win Rate by Year')
ax.legend(fontsize=9)

plt.suptitle('3PT Shooting vs Win Rate: Tournament vs Non-Tournament Over Time', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 2: Does Tournament Success Matter?

Does higher tournament success correspond to better 3-point shooting and higher win rates? Comparing across stages from teams that missed the tournament entirely to champions.

In [ ]:
# --- Graph 3: 3PT Shooting vs Win Rate by Tournament Stage ---
round_labels = {0: 'No Tourney', 1: 'First Four', 2: 'R64', 3: 'R32', 4: 'S16', 5: 'E8', 6: 'F4', 7: 'Final', 8: 'Champion'}
stage_order  = ['No Tourney', 'First Four', 'R64', 'R32', 'S16', 'E8', 'F4', 'Final', 'Champion']
stage_palette = dict(zip(stage_order, sns.color_palette('viridis', len(stage_order))))

plot_df = combined_data.copy()
plot_df['Stage'] = pd.Categorical(plot_df['tournament_round'].map(round_labels), categories=stage_order, ordered=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: 3PT% vs win_pct scatter coloured by stage
ax = axes[0]
for stage in stage_order:
    sub = plot_df[plot_df['Stage'] == stage]
    if len(sub) == 0:
        continue
    ax.scatter(sub['3PT%'], sub['win_pct'], color=stage_palette[stage], alpha=0.5, s=15, label=stage)
ax.set_xlabel('3PT FG%')
ax.set_ylabel('Win Rate (%)')
ax.set_title('3PT% vs Win Rate by Stage')
ax.legend(fontsize=7, markerscale=1.5)

# Panel 2: avg 3PT% by stage (box plot)
ax = axes[1]
sns.boxplot(x='Stage', y='3PT%', data=plot_df, order=stage_order, ax=ax, palette='viridis')
means = plot_df.groupby('Stage', observed=True)['3PT%'].mean().reindex(stage_order)
ax.plot(range(len(stage_order)), means.values, 'D--', color='black', markersize=5, label='Mean')
ax.set_xlabel('')
ax.set_ylabel('3PT FG%')
ax.set_title('3PT% by Tournament Stage')
ax.tick_params(axis='x', rotation=35)
ax.legend(fontsize=9)

# Panel 3: avg 3PTR by stage (box plot)
ax = axes[2]
sns.boxplot(x='Stage', y='3PTR', data=plot_df, order=stage_order, ax=ax, palette='viridis')
means = plot_df.groupby('Stage', observed=True)['3PTR'].mean().reindex(stage_order)
ax.plot(range(len(stage_order)), means.values, 'D--', color='black', markersize=5, label='Mean')
ax.set_xlabel('')
ax.set_ylabel('3PT Attempt Rate (%)')
ax.set_title('3PTR by Tournament Stage')
ax.tick_params(axis='x', rotation=35)
ax.legend(fontsize=9)

plt.suptitle('3PT Shooting vs Win Rate by Tournament Stage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Graph 4: 3-Point Shooting vs Win Rate (by Tournament Status) ---
from scipy import stats

cols   = ['3PT%', '3PTR', 'three_value', 'three_pct_net']
labels = ['3PT FG%', '3PT Attempt Rate', '3PT Value Score', '3PT% Net (Off - Def)']

palette = {'Non-Tournament': '#d9534f', 'Tournament': '#5cb85c'}
combined_data['Tournament'] = combined_data['made_tournament'].map({0: 'Non-Tournament', 1: 'Tournament'})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, col, label in zip(axes, cols, labels):
    for status, color in palette.items():
        subset = combined_data[combined_data['Tournament'] == status]
        ax.scatter(subset[col], subset['win_pct'],
                   color=color, alpha=0.4, s=15, label=status)
        slope, intercept, r, p, _ = stats.linregress(subset[col], subset['win_pct'])
        x_line = np.linspace(subset[col].min(), subset[col].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=1.8,
                label=f'{status} r={r:.2f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Win %')
    ax.set_title(f'{label} vs Win Rate')
    ax.legend(fontsize=8)

plt.suptitle('3-Point Shooting vs Win Rate (Tournament vs Non-Tournament)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()